# 21 — Report Assets: North West Initial Target Intelligence Report v1

This notebook turns the consolidated target-review outputs into a professional report asset pack.

It is deliberately **not** the final report. It is an asset factory: it produces report-ready tables, appendix tables, charts, optional maps, and an asset manifest. The final report should be assembled in DOCX/PDF or slides using these exported assets.

Default scope: **North West**, with optional all-region/national context assets where available.

## 21.1 Setup and paths

Expected structure:

```text
Electoral_Tribes/
  data/
    processed/
      target_review_pack_v1/
      target_model_v2/
      report_assets_v1/
    geography/
      boundaries/
        <ward boundary file>.gpkg/.shp/.geojson  # optional, must include WD25CD
  notebooks/
```

In [13]:
from pathlib import Path
from datetime import datetime
import textwrap
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")

NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
BOUNDARY_DIR = GEOGRAPHY_DIR

INPUT_DIRS = [
    PROCESSED_DIR / "target_review_pack_v1",
    PROCESSED_DIR / "target_model_v2",
    PROCESSED_DIR / "atlas_outputs_v1",
    PROCESSED_DIR / "aggregations_v1",
    PROCESSED_DIR,
]

REPORT_ASSET_DIR = PROCESSED_DIR / "report_assets_v1"
TABLE_DIR = REPORT_ASSET_DIR / "tables"
CHART_DIR = REPORT_ASSET_DIR / "charts"
MAP_DIR = REPORT_ASSET_DIR / "maps"
APPENDIX_DIR = REPORT_ASSET_DIR / "appendices"
MANIFEST_DIR = REPORT_ASSET_DIR / "manifest"

for d in [REPORT_ASSET_DIR, TABLE_DIR, CHART_DIR, MAP_DIR, APPENDIX_DIR, MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Report assets:", REPORT_ASSET_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Report assets: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1


## 21.2 Report configuration and style

The palette is deliberately restrained: navy for the report identity, muted lane colours for strategic interpretation, and simple party colours for charts.

In [14]:
REPORT_TITLE = "North West Initial Target Intelligence Report"
REPORT_SUBTITLE = "Electoral Tribe Model v1.0 — Report Asset Pack"
REPORT_VERSION = "v1.0"
REPORT_DATE = datetime.today().strftime("%Y-%m-%d")

GENERATE_MAPS_IF_BOUNDARIES_AVAILABLE = True

NAVY = "#112A46"
MID_BLUE = "#2B5C88"
LIGHT_GREY = "#F3F5F7"
DARK_GREY = "#3A3A3A"

LANE_ORDER = [
    "Clean Opportunity",
    "Caveated Opportunity",
    "Breakthrough Build",
    "Long-Term Demographic Build",
    "Monitor",
]

LANE_COLORS = {
    "Clean Opportunity": "#1B7837",
    "Caveated Opportunity": "#E66101",
    "Breakthrough Build": "#5E3C99",
    "Long-Term Demographic Build": "#2C7FB8",
    "Monitor": "#BDBDBD",
}

PARTY_COLORS = {
    "lab": "#D73027",
    "con": "#2166AC",
    "ld": "#FDB863",
    "green": "#1A9850",
    "reform_ukip_brexit": "#7B3294",
    "independent": "#4D4D4D",
    "other": "#969696",
    "sdp": "#08306B",
}

CLUSTER_COLORS = {
    "Student & Transient Youth": "#F4A6C8",
    "Rooted Older Homeowners": "#6A3D9A",
    "Stable Suburban Professionals": "#1F78B4",
    "Cosmopolitan Young Professional Core": "#66C2A5",
    "Settled Working Families / Skilled Trades Suburbs": "#FDBF6F",
    "Settled Diverse Urban Communities": "#33A02C",
    "Post-Industrial Estates / Deprived Working Communities": "#E31A1C",
}

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 220
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 9

manifest_rows = []

def add_manifest(filename, asset_type, report_section, description, path):
    manifest_rows.append({
        "filename": filename,
        "asset_type": asset_type,
        "report_section": report_section,
        "description": description,
        "path": str(path),
        "created_at": datetime.now().isoformat(timespec="seconds"),
    })

## 21.3 Utility functions

In [15]:
def find_input_file(filename, required=True):
    # Find a named input file in expected folders.
    for folder in INPUT_DIRS:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    if required:
        searched = "\n".join(str(d / filename) for d in INPUT_DIRS)
        raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")
    return None


def read_csv_if_exists(filename, required=True):
    path = find_input_file(filename, required=required)
    if path is None:
        print(f"Optional file not found: {filename}")
        return None, None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {filename}: {df.shape} from {path}")
    return df, path


def save_table(df, filename, section, description, folder=TABLE_DIR):
    path = folder / filename
    df.to_csv(path, index=False)
    add_manifest(filename, "table_csv", section, description, path)
    print("Saved table:", path)
    return path


def save_chart(fig, filename, section, description):
    path = CHART_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    add_manifest(filename, "chart_png", section, description, path)
    print("Saved chart:", path)
    return path


def wrap_labels(labels, width=24):
    return ["\n".join(textwrap.wrap(str(label), width=width)) for label in labels]


def normalise_bool(s):
    return s.fillna(False).astype(str).str.lower().isin(["true", "1", "yes", "y"])


def pct(x):
    return f"{x:.1f}%" if pd.notna(x) else ""

## 21.4 Load core datasets

This notebook expects outputs from Notebook 20. It also loads all-region summaries if available.

In [16]:
nw_review, _ = read_csv_if_exists("north_west_consolidated_target_review_v1.csv")
nw_council, _ = read_csv_if_exists("north_west_council_briefing_summary_v1.csv")
nw_map_ready, _ = read_csv_if_exists("north_west_target_review_map_ready_v1.csv", required=False)

clean_wards, _ = read_csv_if_exists("north_west_clean_opportunity_wards_v1.csv", required=False)
caveated_wards, _ = read_csv_if_exists("north_west_caveated_opportunity_wards_v1.csv", required=False)
breakthrough_wards, _ = read_csv_if_exists("north_west_breakthrough_build_wards_v1.csv", required=False)
demographic_build_wards, _ = read_csv_if_exists("north_west_long_term_demographic_build_wards_v1.csv", required=False)

all_review, _ = read_csv_if_exists("all_available_consolidated_target_review_v1.csv", required=False)
all_region_summary, _ = read_csv_if_exists("all_available_region_review_summary_v1.csv", required=False)
cluster_key, _ = read_csv_if_exists("k7_cluster_interpretation_key_v1.csv", required=False)

print("North West rows:", len(nw_review))
print("North West councils:", nw_review["LAD25NM"].nunique() if "LAD25NM" in nw_review.columns else "missing")

Loaded north_west_consolidated_target_review_v1.csv: (825, 47) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\north_west_consolidated_target_review_v1.csv
Loaded north_west_council_briefing_summary_v1.csv: (35, 23) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\north_west_council_briefing_summary_v1.csv
Loaded north_west_target_review_map_ready_v1.csv: (825, 21) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\north_west_target_review_map_ready_v1.csv
Loaded north_west_clean_opportunity_wards_v1.csv: (49, 47) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\north_west_clean_opportunity_wards_v1.csv
Loaded north_west_caveated_opportunity_wards_v1.csv: (24, 47) from c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_review_pack_v1\north_west_caveated_opportunity_wards_v1.csv
Loaded north_west_breakthrough_build_wards_v1.csv: (120, 47

## 21.5 Validation and headline metrics

These numbers feed the executive summary and report opening page.

In [17]:
for col in [
    "is_clean_watchlist", "is_caveated_watchlist", "is_breakthrough_complacency",
    "is_demographic_build", "is_top100_watchlist", "has_major_caveat"
]:
    if col in nw_review.columns:
        nw_review[col] = normalise_bool(nw_review[col])
    if all_review is not None and col in all_review.columns:
        all_review[col] = normalise_bool(all_review[col])

headline = {
    "report_title": REPORT_TITLE,
    "report_version": REPORT_VERSION,
    "report_date": REPORT_DATE,
    "north_west_ward_count": len(nw_review),
    "north_west_council_count": nw_review["LAD25NM"].nunique(),
    "clean_opportunity_count": int(nw_review.get("is_clean_watchlist", pd.Series(False)).sum()),
    "caveated_opportunity_count": int(nw_review.get("is_caveated_watchlist", pd.Series(False)).sum()),
    "breakthrough_build_count": int(nw_review.get("is_breakthrough_complacency", pd.Series(False)).sum()),
    "demographic_build_count": int(nw_review.get("is_demographic_build", pd.Series(False)).sum()),
    "major_caveat_count": int(nw_review.get("has_major_caveat", pd.Series(False)).sum()),
    "mean_score": round(float(nw_review["initial_watchlist_score"].mean()), 2),
    "max_score": round(float(nw_review["initial_watchlist_score"].max()), 2),
}
headline_df = pd.DataFrame([headline])
save_table(headline_df, "report_headline_metrics_v1.csv", "Executive Summary", "Core headline statistics for the report.")
headline_df

Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\tables\report_headline_metrics_v1.csv


,report_title,report_version,report_date,north_west_ward_count,north_west_council_count,clean_opportunity_count,caveated_opportunity_count,breakthrough_build_count,demographic_build_count,major_caveat_count,mean_score,max_score
0,North West Initial Target Intelligence Report,v1.0,2026-05-27,825,35,49,24,128,150,154,53.18,79.76


## 21.6 Report-ready tables

These are concise tables for the main report. Full versions are exported later as appendices.

In [18]:
def top_report_table(df, n=12, sort_col="initial_watchlist_score"):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    cols = [
        "LAD25NM", "WD25NM", "strategic_lane", "initial_watchlist_score",
        "dominant_cluster_name", "latest_election_top_party_bucket",
        "demographic_relevance_score", "electoral_opportunity_score", "political_openness_score",
        "breakthrough_complacency_score", "has_major_caveat", "data_confidence_note"
    ]
    cols = [c for c in cols if c in df.columns]
    return df.sort_values(sort_col, ascending=False).head(n)[cols].copy()

report_tables = {}
report_tables["top_clean_opportunity"] = top_report_table(clean_wards if clean_wards is not None else nw_review[nw_review["is_clean_watchlist"]], 15)
report_tables["top_caveated_opportunity"] = top_report_table(caveated_wards if caveated_wards is not None else nw_review[nw_review["is_caveated_watchlist"]], 12)
report_tables["top_breakthrough_build"] = top_report_table(breakthrough_wards if breakthrough_wards is not None else nw_review[nw_review["is_breakthrough_complacency"]], 15, "breakthrough_complacency_score")
report_tables["top_demographic_build"] = top_report_table(demographic_build_wards if demographic_build_wards is not None else nw_review[nw_review["is_demographic_build"]], 15, "demographic_relevance_score")

council_cols = [
    "LAD25NM", "ward_count", "mean_score", "max_score",
    "clean_opportunity_count", "caveated_opportunity_count",
    "breakthrough_build_count", "demographic_build_count", "strategic_interpretation"
]
council_cols = [c for c in council_cols if c in nw_council.columns]
report_tables["council_briefing_leading"] = nw_council.sort_values(
    ["clean_opportunity_count", "breakthrough_build_count", "max_score"], ascending=[False, False, False]
).head(20)[council_cols].copy()

if all_region_summary is not None:
    region_cols = ["analysis_region", "ward_count", "mean_score", "max_score", "clean_opportunity_count", "caveated_opportunity_count", "breakthrough_build_count", "demographic_build_count", "top100_count"]
    region_cols = [c for c in region_cols if c in all_region_summary.columns]
    report_tables["all_region_summary"] = all_region_summary.sort_values("top100_count", ascending=False)[region_cols].copy()

if cluster_key is not None:
    report_tables["cluster_interpretation_key"] = cluster_key.copy()

for name, table in report_tables.items():
    if len(table) == 0:
        print("Skipping empty table:", name)
        continue
    save_table(table, f"{name}_report_table_v1.csv", "Report Tables", f"Report-ready table: {name.replace('_', ' ')}")

Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\tables\top_clean_opportunity_report_table_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\tables\top_caveated_opportunity_report_table_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\tables\top_breakthrough_build_report_table_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\tables\top_demographic_build_report_table_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\tables\council_briefing_leading_report_table_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\tables\all_region_summary_report_table_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\tables\cluster_interpretation_key_report_table_v1.csv


## 21.7 Appendix tables

These are full lists for appendices and manual review, not main-report tables.

In [19]:
appendix_outputs = {
    "north_west_all_review_rows_v1.csv": nw_review,
    "north_west_clean_opportunity_full_v1.csv": clean_wards if clean_wards is not None else nw_review[nw_review["is_clean_watchlist"]],
    "north_west_caveated_opportunity_full_v1.csv": caveated_wards if caveated_wards is not None else nw_review[nw_review["is_caveated_watchlist"]],
    "north_west_breakthrough_build_full_v1.csv": breakthrough_wards if breakthrough_wards is not None else nw_review[nw_review["is_breakthrough_complacency"]],
    "north_west_demographic_build_full_v1.csv": demographic_build_wards if demographic_build_wards is not None else nw_review[nw_review["is_demographic_build"]],
    "north_west_council_briefing_summary_full_v1.csv": nw_council,
}
if all_review is not None:
    appendix_outputs["all_available_consolidated_review_full_v1.csv"] = all_review
if all_region_summary is not None:
    appendix_outputs["all_available_region_review_summary_full_v1.csv"] = all_region_summary

for filename, df in appendix_outputs.items():
    if df is None or len(df) == 0:
        continue
    save_table(df, filename, "Appendix", f"Full appendix table: {filename}", folder=APPENDIX_DIR)

Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\appendices\north_west_all_review_rows_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\appendices\north_west_clean_opportunity_full_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\appendices\north_west_caveated_opportunity_full_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\appendices\north_west_breakthrough_build_full_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\appendices\north_west_demographic_build_full_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\appendices\north_west_council_briefing_summary_full_v1.csv
Saved table: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\appendices\all_available_consolidated_review_full_v1.csv
Saved table: c:\Us

## 21.8 Charts

Charts are PNGs for direct report use. The report should use a small number of these rather than every possible chart.

In [20]:
# Chart 1: strategic lane counts
lane_counts = nw_review["strategic_lane"].fillna("Monitor").value_counts().reindex(LANE_ORDER).dropna()
fig, ax = plt.subplots(figsize=(8.5, 4.8))
colors = [LANE_COLORS.get(lane, "#BDBDBD") for lane in lane_counts.index]
ax.bar(lane_counts.index, lane_counts.values, color=colors)
ax.set_title("North West wards by strategic lane", color=NAVY, weight="bold")
ax.set_ylabel("Ward count")
ax.set_xticklabels(wrap_labels(lane_counts.index, width=18), rotation=0)
ax.spines[["top", "right"]].set_visible(False)
for i, v in enumerate(lane_counts.values):
    ax.text(i, v + max(lane_counts.values)*0.01, str(int(v)), ha="center", va="bottom", fontsize=9)
save_chart(fig, "chart_01_north_west_strategic_lane_counts_v1.png", "Executive Summary", "North West ward counts by strategic lane.")

# Chart 2: top councils by clean opportunity count
if "clean_opportunity_count" in nw_council.columns:
    top = nw_council.sort_values(["clean_opportunity_count", "max_score"], ascending=[False, False]).head(15)
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.barh(top["LAD25NM"][::-1], top["clean_opportunity_count"][::-1], color=LANE_COLORS["Clean Opportunity"])
    ax.set_title("Councils with most clean opportunity wards", color=NAVY, weight="bold")
    ax.set_xlabel("Clean opportunity ward count")
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_02_top_councils_clean_opportunity_v1.png", "Council Intelligence", "Top councils by clean opportunity ward count.")

# Chart 3: top councils by breakthrough build count
if "breakthrough_build_count" in nw_council.columns:
    top = nw_council.sort_values(["breakthrough_build_count", "max_score"], ascending=[False, False]).head(15)
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.barh(top["LAD25NM"][::-1], top["breakthrough_build_count"][::-1], color=LANE_COLORS["Breakthrough Build"])
    ax.set_title("Councils with most breakthrough-build wards", color=NAVY, weight="bold")
    ax.set_xlabel("Breakthrough-build ward count")
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_03_top_councils_breakthrough_build_v1.png", "Council Intelligence", "Top councils by breakthrough-build ward count.")

# Chart 4: all-region top 100 distribution
if all_region_summary is not None and "top100_count" in all_region_summary.columns:
    reg = all_region_summary.sort_values("top100_count", ascending=True)
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    ax.barh(reg["analysis_region"], reg["top100_count"], color=MID_BLUE)
    ax.set_title("All-available top 100 watchlist distribution by region", color=NAVY, weight="bold")
    ax.set_xlabel("Top 100 ward count")
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_04_all_available_top100_by_region_v1.png", "National Signal", "All-available top 100 watchlist distribution by analysis region.")

# Chart 5: score distribution by strategic lane
fig, ax = plt.subplots(figsize=(8.5, 5.2))
for lane in LANE_ORDER:
    s = nw_review.loc[nw_review["strategic_lane"].eq(lane), "initial_watchlist_score"].dropna()
    if len(s) == 0:
        continue
    ax.hist(s, bins=18, alpha=0.55, label=lane, color=LANE_COLORS.get(lane, "#BDBDBD"))
ax.set_title("Initial watchlist score distribution by lane", color=NAVY, weight="bold")
ax.set_xlabel("Initial watchlist score")
ax.set_ylabel("Ward count")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
save_chart(fig, "chart_05_score_distribution_by_lane_v1.png", "Model Review", "Score distribution by strategic lane.")

# Chart 6: latest top party composition by lane
if "latest_election_top_party_bucket" in nw_review.columns:
    party_lane = pd.crosstab(nw_review["strategic_lane"].fillna("Monitor"), nw_review["latest_election_top_party_bucket"].fillna("unknown")).reindex(LANE_ORDER).fillna(0)
    preferred = [c for c in ["lab", "con", "ld", "green", "reform_ukip_brexit", "independent", "other", "sdp"] if c in party_lane.columns]
    party_lane = party_lane[preferred]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    bottom = np.zeros(len(party_lane))
    for party in party_lane.columns:
        vals = party_lane[party].values
        ax.bar(party_lane.index, vals, bottom=bottom, label=party, color=PARTY_COLORS.get(party, "#BDBDBD"))
        bottom += vals
    ax.set_title("Latest top party composition by strategic lane", color=NAVY, weight="bold")
    ax.set_ylabel("Ward count")
    ax.set_xticklabels(wrap_labels(party_lane.index, width=16), rotation=0)
    ax.legend(frameon=False, fontsize=8, ncol=2)
    ax.spines[["top", "right"]].set_visible(False)
    save_chart(fig, "chart_06_latest_top_party_by_lane_v1.png", "Strategic Lanes", "Latest top-party composition by strategic lane.")

Saved chart: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\charts\chart_01_north_west_strategic_lane_counts_v1.png
Saved chart: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\charts\chart_02_top_councils_clean_opportunity_v1.png
Saved chart: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\charts\chart_03_top_councils_breakthrough_build_v1.png
Saved chart: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\charts\chart_04_all_available_top100_by_region_v1.png
Saved chart: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\charts\chart_05_score_distribution_by_lane_v1.png
Saved chart: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\charts\chart_06_latest_top_party_by_lane_v1.png


## 21.9 Optional map generation

Maps are generated only if a ward boundary file is available. The file must include `WD25CD` or a recognisable equivalent. Place it in:

```text
data/geography/boundaries/
```

Recommended assets:

1. Strategic lane map
2. Initial watchlist score map
3. Breakthrough/complacency score map
4. Major caveat map

In [21]:
def find_boundary_file():
    if not BOUNDARY_DIR.exists():
        return None
    candidates = []
    for pattern in ["*.gpkg", "*.shp", "*.geojson", "*.json"]:
        candidates.extend(sorted(BOUNDARY_DIR.glob(pattern)))
    preferred = [p for p in candidates if "ward" in p.name.lower() or "wd25" in p.name.lower()]
    return preferred[0] if preferred else (candidates[0] if candidates else None)

boundary_file = find_boundary_file()
print("Boundary file detected:", boundary_file)

if GENERATE_MAPS_IF_BOUNDARIES_AVAILABLE and boundary_file is not None:
    try:
        import geopandas as gpd
        MAPS_AVAILABLE = True
    except Exception as e:
        print("Geopandas unavailable; skipping maps.")
        print(e)
        MAPS_AVAILABLE = False
else:
    MAPS_AVAILABLE = False

print("Maps available:", MAPS_AVAILABLE)

Boundary file detected: c:\Users\keena\Documents\Electoral_Tribes\data\geography\Wards_May_2025_Boundaries_UK_BGC.gpkg
Maps available: True


In [22]:
def generate_maps_if_possible():
    if not MAPS_AVAILABLE:
        print("Skipping maps: boundary file or geopandas unavailable.")
        return

    wards_gdf = gpd.read_file(boundary_file)
    print("Boundary columns:", wards_gdf.columns.tolist())

    if "WD25CD" not in wards_gdf.columns:
        candidates = [c for c in wards_gdf.columns if c.upper() == "WD25CD" or "WD25CD" in c.upper()]
        if candidates:
            wards_gdf = wards_gdf.rename(columns={candidates[0]: "WD25CD"})
        else:
            raise ValueError("Boundary file does not contain WD25CD or a recognisable equivalent.")

    map_df = nw_review.copy()
    map_df["WD25CD"] = map_df["WD25CD"].astype(str).str.strip()
    wards_gdf["WD25CD"] = wards_gdf["WD25CD"].astype(str).str.strip()
    gdf = wards_gdf.merge(map_df, on="WD25CD", how="inner")
    print("Mapped wards:", len(gdf))

    if len(gdf) == 0:
        print("No boundary rows matched. Map generation skipped.")
        return

    try:
        gdf = gdf.to_crs(27700)
    except Exception:
        pass

    # Map 1: strategic lane
    fig, ax = plt.subplots(figsize=(10, 11))
    gdf["_lane_color"] = gdf["strategic_lane"].map(LANE_COLORS).fillna("#D9D9D9")
    gdf.plot(ax=ax, color=gdf["_lane_color"], linewidth=0.08, edgecolor="white")
    ax.set_axis_off()
    ax.set_title("North West strategic opportunity lanes", color=NAVY, weight="bold", fontsize=16)
    handles = [Patch(facecolor=LANE_COLORS[l], label=l) for l in LANE_ORDER if l in set(gdf["strategic_lane"].dropna())]
    ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8)
    p = MAP_DIR / "map_01_north_west_strategic_lanes_v1.png"
    fig.savefig(p, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    add_manifest(p.name, "map_png", "Strategic Lanes", "North West wards coloured by strategic lane.", p)

    # Map 2: score
    fig, ax = plt.subplots(figsize=(10, 11))
    gdf.plot(ax=ax, column="initial_watchlist_score", cmap="YlOrRd", legend=True, linewidth=0.08, edgecolor="white", missing_kwds={"color": "#F0F0F0"})
    ax.set_axis_off()
    ax.set_title("North West initial watchlist score", color=NAVY, weight="bold", fontsize=16)
    p = MAP_DIR / "map_02_north_west_initial_watchlist_score_v1.png"
    fig.savefig(p, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    add_manifest(p.name, "map_png", "Score Map", "North West wards by initial watchlist score.", p)

    # Map 3: breakthrough score
    if "breakthrough_complacency_score" in gdf.columns:
        fig, ax = plt.subplots(figsize=(10, 11))
        gdf.plot(ax=ax, column="breakthrough_complacency_score", cmap="Purples", legend=True, linewidth=0.08, edgecolor="white", missing_kwds={"color": "#F0F0F0"})
        ax.set_axis_off()
        ax.set_title("North West breakthrough/complacency potential", color=NAVY, weight="bold", fontsize=16)
        p = MAP_DIR / "map_03_north_west_breakthrough_complacency_v1.png"
        fig.savefig(p, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        add_manifest(p.name, "map_png", "Breakthrough Build", "North West wards by breakthrough/complacency score.", p)

    # Map 4: caveats
    if "has_major_caveat" in gdf.columns:
        fig, ax = plt.subplots(figsize=(10, 11))
        caveat_colors = {True: "#E66101", False: "#BDBDBD"}
        gdf["_caveat_color"] = gdf["has_major_caveat"].fillna(False).map(caveat_colors)
        gdf.plot(ax=ax, color=gdf["_caveat_color"], linewidth=0.08, edgecolor="white")
        ax.set_axis_off()
        ax.set_title("North West data caveat map", color=NAVY, weight="bold", fontsize=16)
        handles = [Patch(facecolor="#E66101", label="Major caveat"), Patch(facecolor="#BDBDBD", label="No major caveat")]
        ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=8)
        p = MAP_DIR / "map_04_north_west_major_caveats_v1.png"
        fig.savefig(p, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        add_manifest(p.name, "map_png", "Caveats", "North West wards by major caveat status.", p)

generate_maps_if_possible()

Boundary columns: ['WD25CD', 'WD25NM', 'WD25NMW', 'LAD25CD', 'LAD25NM', 'LAD25NMW', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'GlobalID', 'geometry']
Mapped wards: 825


## 21.10 Asset manifest

The manifest is the control sheet for report assembly.

In [ ]:
manifest = pd.DataFrame(manifest_rows)
manifest_path = MANIFEST_DIR / "report_asset_manifest_v1.csv"
manifest.to_csv(manifest_path, index=False)
print("Saved manifest:", manifest_path)
manifest

Saved manifest: c:\Users\keena\Documents\Electoral_Tribes\data\processed\report_assets_v1\manifest\report_asset_manifest_v1.csv


,filename,asset_type,report_section,description,path,created_at
0,report_headline_metrics_v1.csv,table_csv,Executive Summary,Core headline statistics for the report.,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
1,top_clean_opportunity_report_table_v1.csv,table_csv,Report Tables,Report-ready table: top clean opportunity,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
2,top_caveated_opportunity_report_table_v1.csv,table_csv,Report Tables,Report-ready table: top caveated opportunity,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
3,top_breakthrough_build_report_table_v1.csv,table_csv,Report Tables,Report-ready table: top breakthrough build,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
4,top_demographic_build_report_table_v1.csv,table_csv,Report Tables,Report-ready table: top demographic build,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
5,council_briefing_leading_report_table_v1.csv,table_csv,Report Tables,Report-ready table: council briefing leading,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
6,all_region_summary_report_table_v1.csv,table_csv,Report Tables,Report-ready table: all region summary,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
7,cluster_interpretation_key_report_table_v1.csv,table_csv,Report Tables,Report-ready table: cluster interpretation key,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
8,north_west_all_review_rows_v1.csv,table_csv,Appendix,Full appendix table: north_west_all_review_row...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45
9,north_west_clean_opportunity_full_v1.csv,table_csv,Appendix,Full appendix table: north_west_clean_opportun...,c:\Users\keena\Documents\Electoral_Tribes\data...,2026-05-27T16:39:45


: 

## 21.11 Final report build sequence

After this notebook runs:

1. Use `report_assets_v1/tables/` for main report tables.
2. Use `report_assets_v1/appendices/` for full review lists.
3. Use `report_assets_v1/charts/` for report graphics.
4. Use `report_assets_v1/maps/` if boundary data was supplied.
5. Use `report_assets_v1/manifest/report_asset_manifest_v1.csv` as the asset checklist.

The final report should be assembled manually as a designed DOCX/PDF using these assets.